In [ ]:
import os
from pathlib import Path


from aereo.backends import LocalProcessBackend
from aereo.client import AereoClient
from aereo.pipeline import ExtractionJob


DRY_RUN = os.environ.get("DRY_RUN", "false").lower() in ("1", "true", "yes")
job = ExtractionJob.load_from_config(
    config_dir=(Path(".").parent / "config").resolve(),
    config_name="job_viirs.yaml",
)

In [ ]:
def run_pipeline(job: ExtractionJob) -> None:
    """Run search → prepare → extract for a validated job.

    Args:
        job: The validated ``ExtractionJob`` to execute.
    """
    client = AereoClient()

    # Search
    print("\n🔍 Searching...")
    search_results = client.search(job.search)
    print(f"✓ Found {len(search_results)} scenes")

    if search_results.empty:
        print("No results; skipping prepare/extract.")
        return

    # Prepare
    print("\n📦 Preparing tasks...")
    tasks = client.prepare_tasks(
        search_results=search_results,
        job=job,
        cells_per_task=20,
    )
    print(f"✓ Prepared {len(tasks)} tasks")

    # Extract
    print("\n⛏️ Extracting...")
    backend = LocalProcessBackend(max_workers=2)
    artifacts = client.execute_tasks(tasks, backend=backend)
    print(f"✓ Extracted {len(artifacts)} artifacts")
    return artifacts

In [ ]:
artifacts = run_pipeline(job)

In [ ]:
artifacts.iloc[120].uri

In [ ]:
import rioxarray

rioxarray.open_rasterio(artifacts.iloc[120].uri).plot()